# Huawei Technology Lab 3 — MindSpore 2D Segmentation
Learn image → mask → Dice score with a small U-Net-style network written in Huawei MindSpore.

Synthetic images and masks are used so the segmentation workflow can be learned without patient data.

**Educational prototype — not for clinical diagnosis or treatment.**

In [ ]:
import numpy as np
import mindspore as ms
from mindspore import nn, ops
import mindspore.dataset as ds

ms.set_seed(42)
rng=np.random.default_rng(42)
SIZE=64
yy,xx=np.mgrid[:SIZE,:SIZE]
images=[]; masks=[]
for _ in range(160):
    image=rng.normal(0.2,0.05,(SIZE,SIZE)).astype(np.float32)
    cx,cy=int(rng.integers(16,49)),int(rng.integers(16,49))
    rx,ry=int(rng.integers(5,12)),int(rng.integers(5,12))
    mask=((((xx-cx)/rx)**2+((yy-cy)/ry)**2)<=1).astype(np.float32)
    image=np.clip(image+mask*0.5,0,1)
    images.append(image[None]); masks.append(mask[None])
images=np.stack(images).astype(np.float32); masks=np.stack(masks).astype(np.float32)
train_ds=ds.NumpySlicesDataset((images[:128],masks[:128]),column_names=['image','mask'],shuffle=True).batch(8)
X_test=images[128:]; y_test=masks[128:]
print(images.shape,masks.shape)

## Build a tiny U-Net-style model

In [ ]:
class DoubleConv(nn.Cell):
    def __init__(self,cin,cout):
        super().__init__()
        self.block=nn.SequentialCell(
            nn.Conv2d(cin,cout,3,pad_mode='pad',padding=1,has_bias=True),nn.ReLU(),
            nn.Conv2d(cout,cout,3,pad_mode='pad',padding=1,has_bias=True),nn.ReLU())
    def construct(self,x): return self.block(x)

class TinyUNet(nn.Cell):
    def __init__(self):
        super().__init__()
        self.enc=DoubleConv(1,8); self.pool=nn.MaxPool2d(2,2)
        self.bridge=DoubleConv(8,16)
        self.up=nn.Conv2dTranspose(16,8,2,stride=2,pad_mode='pad',padding=0,has_bias=True)
        self.dec=DoubleConv(16,8)
        self.head=nn.Conv2d(8,1,1,pad_mode='valid',has_bias=True)
    def construct(self,x):
        skip=self.enc(x)
        b=self.bridge(self.pool(skip))
        d=ops.concat((self.up(b),skip),axis=1)
        return self.head(self.dec(d))

net=TinyUNet(); bce=nn.BCEWithLogitsLoss(); optimizer=nn.Adam(net.trainable_params(),learning_rate=0.001)
def forward_fn(x,y):
    logits=net(x); return bce(logits,y),logits
grad_fn=ms.value_and_grad(forward_fn,None,optimizer.parameters,has_aux=True)
def train_step(x,y):
    (loss,logits),grads=grad_fn(x,y); optimizer(grads); return loss

for epoch in range(5):
    losses=[]
    for x,y in train_ds.create_tuple_iterator(): losses.append(float(train_step(x,y).asnumpy()))
    print(f'Epoch {epoch+1}: loss={np.mean(losses):.4f}')

## Evaluate with Dice overlap

In [ ]:
net.set_train(False)
prob=ops.sigmoid(net(ms.Tensor(X_test,ms.float32))).asnumpy()
pred=(prob>=0.5).astype(np.float32)
p=pred.reshape(len(pred),-1); t=y_test.reshape(len(y_test),-1)
intersection=(p*t).sum(axis=1)
dice=((2*intersection+1)/(p.sum(axis=1)+t.sum(axis=1)+1)).mean()
print('Synthetic test Dice:',round(float(dice),3))
ms.save_checkpoint(net,'mindspore_2d_segmenter.ckpt')

## Clinical interpretation checkpoint
Classification answers **what?** Segmentation adds **where?** A Dice score summarizes overlap, but the predicted mask should also be inspected visually. A good score on synthetic data does not establish safety, accuracy, or generalizability on real medical images.